# 👕 FitDiT Virtual Try-On (ComfyUI Colab)
**FitDiT** utilizes Diffusion Transformers (DiT) to achieve high-fidelity virtual try-on with realistic garment folding.

--- 
Developed by **nano**

In [ ]:
#@title 1. Install FitDiT Nodes
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install xformers -r requirements.txt

%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
!git clone https://github.com/BoyuanJiang/FitDiT.git -b FitDiT-ComfyUI FitDiT

%cd /content/ComfyUI/custom_nodes/FitDiT
!pip install -r requirements.txt

In [ ]:
#@title 2. Download FitDiT Checkpoints
!mkdir -p /content/ComfyUI/models/FitDiT_models
%cd /content/ComfyUI/models/FitDiT_models
!apt-get -y install -qq aria2

# Main FitDiT Model
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/BoyuanJiang/FitDiT/resolve/main/pytorch_model.bin -o fitdit_model.bin

# CLIP Models required by FitDiT
!mkdir -p /content/ComfyUI/models/clip
%cd /content/ComfyUI/models/clip
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/pytorch_model.bin -o clip_large.bin

In [ ]:
#@title 3. Launch
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess, threading, time, socket
def iframe_thread(port):
  while True:
      time.sleep(0.5)
      if socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect_ex(('127.0.0.1', port)) == 0: break
  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    if "trycloudflare.com " in line.decode(): print("URL:", line.decode()[line.decode().find("http"):])

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
%cd /content/ComfyUI
!python main.py --dont-print-server